In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
import os

DATA_PATH = "/kaggle/input/datasets/abhishekpriya/mit-bih/mit-bih-arrhythmia-database-1.0.0"

records = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".dat"):
        records.append(file[:-4])  # remove .dat

print(records[:10])

Install ECG Library

In [ ]:
!pip install wfdb # Read ECG files easily.

Import Libraries

In [ ]:
import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# Load First ECG Record
record = wfdb.rdrecord(
    DATA_PATH + "/100"
)

In [ ]:
# Load Annotations-----heartbeat labels
ann = wfdb.rdann(
    DATA_PATH + "/100",
    "atr"
)


In [ ]:
# Extract ECG Signal-----Use one ECG channel.
ecg = record.p_signal[:,0]

print(ecg.shape)

Plot Raw ECG

In [ ]:
plt.figure(figsize=(15,4))

plt.plot(ecg[:3000])

plt.title("Raw ECG")

plt.show()

Inspect Labels -- Understand heartbeat classes.

In [ ]:
print(ann.symbol[:20])

Inspect R-Peaks---Locations of heartbeats.

In [ ]:
print(ann.sample[:20])

Normalize Signal

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

ecg_norm = scaler.fit_transform(
    ecg.reshape(-1,1)
)

ecg_norm = ecg_norm.flatten()

Remove Baseline Wander

Purpose:

Remove slow drift caused by breathing and movement.

In [ ]:
from scipy.signal import medfilt

baseline = medfilt(
    ecg_norm,
    kernel_size=201
)

ecg_baseline_removed = (
    ecg_norm - baseline
)

Remove High-Frequency Noise

Purpose:

Remove unwanted signal noise.

In [ ]:
from scipy.signal import butter,filtfilt

def lowpass_filter(
    data,
    cutoff=40,
    fs=360,
    order=4
):

    nyquist = 0.5*fs

    normal_cutoff = cutoff/nyquist

    b,a = butter(
        order,
        normal_cutoff,
        btype='low'
    )

    return filtfilt(
        b,a,data
    )

In [ ]:
filtered_ecg = lowpass_filter(
    ecg_baseline_removed
)

Plot Clean ECG

Purpose:

Verify preprocessing worked.

In [ ]:
plt.figure(figsize=(15,4))

plt.plot(filtered_ecg[:3000])

plt.title("Filtered ECG")

plt.show()

Heartbeat Segmentation

Purpose:

Extract individual heartbeats.

In [ ]:
segments = []
labels = []

window = 180

In [ ]:
for peak,label in zip(
        ann.sample,
        ann.symbol):

    start = peak-window
    end = peak+window

    if start < 0:
        continue

    if end > len(filtered_ecg):
        continue

    beat = filtered_ecg[start:end]

    segments.append(beat)
    labels.append(label)

Create Dataset

Purpose:

Convert heartbeat list into model-ready format.

In [ ]:
X = np.array(segments)

print(X.shape)

Visualize One Heartbeat

Purpose:

See what one training sample looks like.

In [ ]:
plt.figure(figsize=(8,3))

plt.plot(X[0])

plt.title("Single Heartbeat")

plt.show()

Encode Labels

Purpose:

Convert heartbeat symbols into numbers.

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(labels)

print(np.unique(y))

Train-Test Split

Purpose:

Create training and testing data.

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Convert To PyTorch Tensors

Purpose:

Prepare data for Transformer.

In [ ]:
import torch

X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test = torch.tensor(
    y_test,
    dtype=torch.long
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

X_train = X_train.to(device)
X_test = X_test.to(device)

y_train = y_train.to(device)
y_test = y_test.to(device)

Now i will load all ECG signals


In [ ]:
from scipy.signal import butter, filtfilt

In [ ]:
records = []

for file in os.listdir(DATA_PATH):

    if file.endswith(".dat"):

        record_name = file.replace(".dat","")

        records.append(record_name)

records = sorted(records)

print("Total Records:", len(records))
print(records[:10])

Noise Removal Function

use a Bandpass Filter.

In [ ]:
from scipy.signal import butter, filtfilt

def bandpass_filter(signal,
                    lowcut=0.5,
                    highcut=40,
                    fs=360,
                    order=4):

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order,
                  [low, high],
                  btype='band')

    filtered = filtfilt(b, a, signal)

    return filtered

Why?

Removes:

Baseline drift,
Powerline noise,
Motion artifacts

In [ ]:
all_beats = []
all_labels = []

for rec in records: #Process Every Record

    print("Processing:", rec)

    # Load ECG record
    record = wfdb.rdrecord(
        DATA_PATH + "/" + rec
    )

    # Load annotations
    annotation = wfdb.rdann(
        DATA_PATH + "/" + rec,
        "atr"
    )

    # Extract ECG signal
    ecg = record.p_signal[:,0]

    # Remove noise
    ecg = bandpass_filter(ecg)

    # Get R-peaks and labels
    r_peaks = annotation.sample
    labels = annotation.symbol

    #Heartbeat Segmentation

    for i in range(len(r_peaks)):

        peak = r_peaks[i]

        label = labels[i]

        if peak - 180 < 0:
            continue

        if peak + 180 >= len(ecg):
            continue

        beat = ecg[
            peak-180 :
            peak+180
        ]

        all_beats.append(beat)

        all_labels.append(label)

numpy array

In [ ]:
X = np.array(all_beats)

y = np.array(all_labels)

print(X.shape)

print(y.shape)

In [ ]:
# checking labels
from collections import Counter

print(Counter(y))

Keep Important Classes Only

Most papers use:

N = Normal

V = PVC

A = Atrial

F = Fusion

Q = Unknown

In [ ]:
mapping = {
    'N':'N',
    'L':'N',
    'R':'N',

    'A':'S',
    'a':'S',
    'J':'S',
    'S':'S',

    'V':'V',
    'E':'V',

    'F':'F',

    '/':'Q',
    'f':'Q',
    'Q':'Q'
}

In [ ]:
mask = np.isin(
    all_labels,
    list(mapping.keys())
)

X = np.array(all_beats)[mask]
y = np.array(all_labels)[mask]

Convert to AAMI Classes

In [ ]:
y = np.array(
    [mapping[label] for label in y]
)

In [ ]:
from collections import Counter

print(Counter(y))

| Class | Meaning                  | Samples |
| ----- | ------------------------ | ------: |
| N     | Normal Beat              |  90,337 |
| Q     | Unknown / Unclassifiable |   8,038 |
| V     | Ventricular Beat         |   7,235 |
| S     | Supraventricular Beat    |   2,781 |
| F     | Fusion Beat              |     802 |


Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(np.unique(y))

In [ ]:
# check mapping
print(encoder.classes_)

F → 0

N → 1

Q → 2

S → 3

V → 4

Normalize ECG Beats

In [ ]:
X_norm = []

for beat in X:

    beat = (beat - np.min(beat)) / \
           (np.max(beat) - np.min(beat) + 1e-8)

    X_norm.append(beat)

X = np.array(X_norm)

print(X.shape)

In [ ]:
#saving dataset
np.save("/content/X.npy", X)
np.save("/content/y.npy", y)

print("Saved")

Train (70%) and Remaining (30%)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

Second Split: Validation (20%) and Test (10%)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=1/3,
    random_state=42,
    stratify=y_temp
)

In [ ]:
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Add Channel Dimension for Transformer

In [ ]:
X_train = np.expand_dims(X_train, axis=1)
X_val   = np.expand_dims(X_val, axis=1)
X_test  = np.expand_dims(X_test, axis=1)

print(X_train.shape)


create a folder

In [ ]:
import os

os.makedirs(
    "/kaggle/working/ecg_data",
    exist_ok=True
)

In [ ]:
np.save(
    "/kaggle/working/ecg_data/X_train.npy",
    X_train
)

np.save(
    "/kaggle/working/ecg_data/X_val.npy",
    X_val
)

np.save(
    "/kaggle/working/ecg_data/X_test.npy",
    X_test
)

np.save(
    "/kaggle/working/ecg_data/y_train.npy",
    y_train
)

np.save(
    "/kaggle/working/ecg_data/y_val.npy",
    y_val
)

np.save(
    "/kaggle/working/ecg_data/y_test.npy",
    y_test
)

In [ ]:
os.listdir(
    "/kaggle/working/ecg_data"
)

In [ ]:
from collections import Counter

print("Train:", Counter(y_train))
print("Val:", Counter(y_val))
print("Test:", Counter(y_test))

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

ECG Dataset

In [ ]:
class ECGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return len(self.y)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

Create Dataset

In [ ]:
train_dataset = ECGDataset(
    X_train,
    y_train
)

val_dataset = ECGDataset(
    X_val,
    y_val
)

test_dataset = ECGDataset(
    X_test,
    y_test
)

DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

Verify Shape

In [ ]:
for X_batch, y_batch in train_loader:

    print(X_batch.shape)

    print(y_batch.shape)

    break

Patch Embedding

In [ ]:
class PatchEmbedding(nn.Module):

    def __init__(
        self,
        patch_len=18,
        stride=9,
        embed_dim=128
    ):
        super().__init__()

        self.patch_len = patch_len
        self.stride = stride

        self.proj = nn.Linear(
            patch_len,
            embed_dim
        )

    def forward(self, x):

        B,C,L = x.shape

        x = x.unfold(
            -1,
            self.patch_len,
            self.stride
        )

        P = x.shape[2]

        x = x.reshape(
            B*C,
            P,
            self.patch_len
        )

        x = self.proj(x)

        return x,B,C,P

Transformer Model

In [ ]:
class PatchTST_ECG_Classifier(nn.Module):

    def __init__(
        self,
        num_classes=5
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding()

        self.pos_embedding = nn.Parameter(
            torch.randn(
                1,
                39,
                128
            )
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=8,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                128,
                128
            ),

            nn.GELU(),

            nn.Dropout(
                0.3
            ),
            nn.Linear(
                128,
                64
            ),

            nn.GELU(),

            nn.Dropout(
                0.3
            ),


            nn.Linear(
                64,
                num_classes
            )
        )

    def forward(self,x):

        x,B,C,P = self.patch_embed(x)

        x = x + self.pos_embedding[:,:P,:]

        x = self.transformer(x)

        x = x.mean(dim=1)

        x = x.view(
            B,
            C,
            -1
        )

        x = x.mean(dim=1)

        return self.classifier(x)

Create Model

In [ ]:
model = PatchTST_ECG_Classifier(
    num_classes=5
).to(device)

# ---------- Class Weights ----------

counts = np.bincount(y_train)

weights = 1.0 / np.sqrt(counts)

weights = weights / weights.min()

weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights
)

# ---------- Optimizer ----------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

# ---------- Scheduler ----------

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20,
    eta_min=1e-5
)

Validation Function

This computes:

Accuracy,
Precision,
Recall,
F1

In [ ]:
def evaluate(
    model,
    loader
):

    model.eval()

    preds = []
    labels = []

    with torch.no_grad():

        for X, y in loader:

            X = X.to(device)

            output = model(X)

            pred = torch.argmax(
                output,
                dim=1
            )

            preds.extend(
                pred.cpu().numpy()
            )

            labels.extend(
                y.numpy()
            )

    accuracy = accuracy_score(
        labels,
        preds
    )

    precision = precision_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    recall = recall_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    f1 = f1_score(
        labels,
        preds,
        average='macro',
        zero_division=0
    )

    return (
        accuracy,
        precision,
        recall,
        f1
    )

Training Loop

In [ ]:
EPOCHS = 40

best_f1 = 0
patience = 10
counter = 0

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(
            output,
            y
        )

        # Backpropagation
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    # Learning Rate Scheduler
    scheduler.step()

    acc, prec, rec, f1 = evaluate(
        model,
        val_loader
    )

    print(f"Epoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"Acc: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}")

    if f1 > best_f1:

        best_f1 = f1
        counter = 0

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print("Model Saved")

    else:

        counter += 1

    if counter >= patience:

        print("Early Stopping")
        break

In [ ]:
# Load Best Model
model.load_state_dict(
    torch.load(
        "best_model.pth"
    )
)

In [ ]:
acc,prec,rec,f1 = evaluate(
    model,
    test_loader
)

print("Test Accuracy:",acc)

print("Test Precision:",prec)

print("Test Recall:",rec)

print("Test F1:",f1)

In [ ]:
# Classification Report
preds = []
labels = []

model.eval()

with torch.no_grad():

    for X,y in test_loader:

        X = X.to(device)

        output = model(X)

        pred = output.argmax(1)

        preds.extend(
            pred.cpu().numpy()
        )

        labels.extend(
            y.numpy()
        )

print(
    classification_report(
        labels,
        preds
    )
)